SHROOM FLAN LoRA Final Test-Set Fine-Tuning Runner - Colab/VS Code

Final protocol:
- train LoRA adapters on the full SHROOM dev/validation set,
- use the fixed FLAN LoRA hyperparameters selected from internal-split pilots,
- do not evaluate/select checkpoints on the test set during training,
- evaluate once on the labeled SHROOM test set after fixed-epoch training.

1) Configuration

In [1]:
from pathlib import Path
from datetime import datetime

# Google Drive project folder containing finetune_flan_lora_final_eval.py, src/, data/, participant_kit/.
# Edit this if your Drive folder differs.
DRIVE_PROJECT = Path("/content/drive/MyDrive/thesis_colab/model_experiments_colab")

# Temporary Colab workspace used for actual execution.
LOCAL_PROJECT = Path("/content/model_experiments_colab")

# Where this runner backs up outputs and logs in Google Drive.
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/thesis_colab/outputs_flan_lora_finetune_test_colab_vscode - A100, rerun")

# One timestamped backup folder for this session.
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
DRIVE_RUN_OUTPUT_DIR = DRIVE_OUTPUT_ROOT / RUN_TAG

# Script to run. Upload/copy the provided final-eval script to your Drive project root.
FINETUNE_SCRIPT = "finetune_flan_lora_final_eval_runtime_enhanced.py"

# Dataset paths relative to LOCAL_PROJECT.
# Final protocol: train on all SHROOM dev/validation, evaluate on labeled SHROOM test.
TRAIN_PATH = "data/SHROOM_dev-v2/val.model-agnostic.json"
EVAL_PATH = "data/SHROOM_test-labeled/test.model-agnostic.json"
FINAL_EVAL_ONLY = True
WARMUP_PATH = "data/SHROOM_trial-v1.1/trial-v1.json"
WARMUP_N = 10

# Fixed FLAN LoRA recipe selected from internal-split pilots.
SEED = 42
PREVIEW_N = 2
LABEL_MODE = "soft"
EPOCHS = 4
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.0
WARMUP_RATIO = 0.1
MAX_LENGTH = 512
BEST_METRIC = "rho"  # retained for internal-split mode; ignored for final-eval-only checkpoint choice
FP16 = False          # keep False because fp16 produced unstable/NaN training in the FLAN-small smoke test

# LoRA configuration used in the stronger internal-split pilots.
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = "q,k,v,o"

# Optional smoke-test limits. Leave as None for final reporting.
TRAIN_LIMIT = None
EVAL_LIMIT = None

# Participant scorer controls.
# False = use direct metrics stored in metadata from the labeled test file.
# True = additionally write outputs/predictions/current/test.model-agnostic.json and run participant_kit/score.py.
# For test scoring, the script does NOT pass --is_val.
WRITE_CURRENT = True
RUN_PARTICIPANT_SCORER = True
REFERENCE_DIR = "data/SHROOM_test-labeled"
SCORE_SPLIT = "test"

# Execution controls.
CONTINUE_ON_ERROR = False
SKIP_EXISTING = False
DRY_RUN = False  # First run with True. Then set False when commands look right.

# Optional filters.
# Examples:
# ONLY_CONTAINS = "small"   # one-model smoke test; note this matches both small and base? use ONLY_MODEL_NAMES for exact.
# ONLY_MODEL_NAMES = ["google/flan-t5-small"]
ONLY_CONTAINS = ""
ONLY_MODEL_NAMES = []  # Optional exact model-name list. Leave empty to ignore.

# If True, remove LOCAL_PROJECT/outputs after syncing from Drive.
# Usually leave False when using SKIP_EXISTING.
CLEAR_LOCAL_OUTPUTS = False

print("Drive project:", DRIVE_PROJECT)
print("Local project:", LOCAL_PROJECT)
print("Drive output folder for this run:", DRIVE_RUN_OUTPUT_DIR)
print("Run tag:", RUN_TAG)
print("Final eval only:", FINAL_EVAL_ONLY)
print("Epochs:", EPOCHS)

Drive project: /content/drive/MyDrive/thesis_colab/model_experiments_colab
Local project: /content/model_experiments_colab
Drive output folder for this run: /content/drive/MyDrive/thesis_colab/outputs_flan_lora_finetune_test_colab_vscode - A100, rerun/20260514_164059
Run tag: 20260514_164059
Final eval only: True
Epochs: 4


2) Install/check dependencies and GPU status

In [2]:
import subprocess
import sys
import platform

INSTALL_DEPENDENCIES = True
UNINSTALL_INCOMPATIBLE_TORCHAO = True

if INSTALL_DEPENDENCIES:
    packages = [
        "transformers<5",
        "accelerate",
        "peft",
        "sentencepiece",
        "protobuf<6",
        "scipy",
        "scikit-learn",
        "huggingface_hub",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *packages])

# Some Colab images include torchao 0.10.x, which can conflict with recent PEFT LoRA adapter injection.
# FLAN LoRA does not need torchao, so removing it is the simplest stable fix.
if UNINSTALL_INCOMPATIBLE_TORCHAO:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)

print("Python:", sys.version)
print("Platform:", platform.platform())

try:
    import torch
    print("torch:", torch.__version__)
    print("cuda_available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu_name:", torch.cuda.get_device_name(0))
        props = torch.cuda.get_device_properties(0)
        print("gpu_total_memory_gb:", round(props.total_memory / 1024**3, 2))
        print("bf16_supported:", torch.cuda.is_bf16_supported())
except Exception as exc:
    print("Could not inspect torch/GPU:", repr(exc))

try:
    subprocess.run(["nvidia-smi"], check=False)
except Exception as exc:
    print("nvidia-smi unavailable:", repr(exc))

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
torch: 2.10.0+cu128
cuda_available: True
gpu_name: NVIDIA A100-SXM4-40GB
gpu_total_memory_gb: 39.49
bf16_supported: True


3) Mount Google Drive

In [3]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Could not import/mount google.colab.drive. Are you connected to a Colab runtime?")
    raise

Mounted at /content/drive


4) Sync project from Google Drive to /content

In [4]:
# Rerun this cell whenever you update files in Drive and want /content to use the latest version.
import os
import shutil
from pathlib import Path

if not DRIVE_PROJECT.exists():
    raise FileNotFoundError(
        f"Drive project folder not found: {DRIVE_PROJECT}\n"
        "Create/upload your folder at this path, or edit DRIVE_PROJECT in cell 1."
    )

if LOCAL_PROJECT.exists():
    shutil.rmtree(LOCAL_PROJECT)

ignore = shutil.ignore_patterns(
    ".git",
    ".venv",
    "venv",
    "__pycache__",
    ".pytest_cache",
    ".mypy_cache",
    ".ipynb_checkpoints",
    "wandb",
)

shutil.copytree(DRIVE_PROJECT, LOCAL_PROJECT, ignore=ignore)
os.chdir(LOCAL_PROJECT)

if CLEAR_LOCAL_OUTPUTS:
    outputs_dir = LOCAL_PROJECT / "outputs"
    if outputs_dir.exists():
        shutil.rmtree(outputs_dir)

print("Working directory:", Path.cwd())
print("Top-level files:")
for p in sorted(Path.cwd().iterdir()):
    print("-", p.name)

Working directory: /content/model_experiments_colab
Top-level files:
- data
- finetune_deberta.py
- finetune_deberta_final_eval.py
- finetune_deberta_lora_final_eval_v2.py
- finetune_deberta_lora_v2.py
- finetune_flan_lora.py
- finetune_flan_lora_final_eval.py
- finetune_flan_lora_final_eval_runtime_enhanced.py
- finetune_gemma_lora_final_eval.py
- finetune_gemma_lora_v2.py
- finetune_qwen_lora.py
- finetune_qwen_lora_final_eval.py
- participant_kit
- run_experiment.py
- run_experiment_ootb_eval.py
- run_experiment_ootb_eval_runtime_enhanced.py
- src


5) Sanity-check expected files before running

In [5]:
from pathlib import Path

required_paths = [
    FINETUNE_SCRIPT,
    "src/data.py",
    "src/prompts.py",
    TRAIN_PATH,
    EVAL_PATH,
    WARMUP_PATH,
]

# Participant kit is optional because RUN_PARTICIPANT_SCORER is False by default.
if RUN_PARTICIPANT_SCORER:
    required_paths.extend([
        "participant_kit/check_output.py",
        "participant_kit/score.py",
    ])

missing = [p for p in required_paths if not Path(p).exists()]
if missing:
    print("Missing required files:")
    for p in missing:
        print("-", p)
    raise FileNotFoundError("Project structure check failed.")

print("Project structure looks good.")

Project structure looks good.


6) Final test-set fine-tuning job list: FLAN LoRA size rungs

In [6]:
# Uses the same effective train batch size = 4 logic from the validation pilots.
# XL is included but off by default because you decided to move to final test for small/base/large first.
FINETUNE_JOBS = [
    {
        "model_name": "google/flan-t5-small",
        "run": False,
        "train_batch_size": 2,
        "eval_batch_size": 4,
        "grad_accum_steps": 2,
        "gradient_checkpointing": False,
    },
    {
        "model_name": "google/flan-t5-base",
        "run": False,
        "train_batch_size": 1,
        "eval_batch_size": 2,
        "grad_accum_steps": 4,
        "gradient_checkpointing": False,
    },
    {
        "model_name": "google/flan-t5-large",
        "run": True,
        "train_batch_size": 1,
        "eval_batch_size": 1,
        "grad_accum_steps": 4,
        "gradient_checkpointing": False,
    },
    {
        "model_name": "google/flan-t5-xl",
        "run": True,
        "train_batch_size": 1,
        "eval_batch_size": 1,
        "grad_accum_steps": 4,
        "gradient_checkpointing": False,
    },
]

print("Configured final FLAN LoRA jobs:")
for job in FINETUNE_JOBS:
    eff_bs = job["train_batch_size"] * job["grad_accum_steps"]
    print(
        f"- {job['model_name']} | run={job['run']} | "
        f"train_bs={job['train_batch_size']} | eval_bs={job['eval_batch_size']} | "
        f"grad_accum={job['grad_accum_steps']} | effective_bs={eff_bs} | "
        f"grad_ckpt={job['gradient_checkpointing']}"
    )

Configured final FLAN LoRA jobs:
- google/flan-t5-small | run=False | train_bs=2 | eval_bs=4 | grad_accum=2 | effective_bs=4 | grad_ckpt=False
- google/flan-t5-base | run=False | train_bs=1 | eval_bs=2 | grad_accum=4 | effective_bs=4 | grad_ckpt=False
- google/flan-t5-large | run=True | train_bs=1 | eval_bs=1 | grad_accum=4 | effective_bs=4 | grad_ckpt=False
- google/flan-t5-xl | run=True | train_bs=1 | eval_bs=1 | grad_accum=4 | effective_bs=4 | grad_ckpt=False


7) Optional Hugging Face login

In [7]:
# Usually not needed for FLAN-T5 models.
LOGIN_TO_HF = False

if LOGIN_TO_HF:
    from huggingface_hub import login

    token = None
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None

    if token is None:
        import getpass
        token = getpass.getpass("Paste Hugging Face token: ")

    login(token=token)
    print("Logged into Hugging Face.")
else:
    print("HF login skipped.")

HF login skipped.


8) Runner helpers: filtering, streamed subprocess logs, existing-run checks, and Drive backup

In [8]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path
from datetime import datetime

def safe_filename(text: str) -> str:
    return (
        text.replace("/", "__")
        .replace("\\", "__")
        .replace(":", "_")
        .replace(" ", "_")
    )

def selected_jobs(jobs: list[dict]) -> list[dict]:
    only_contains = ONLY_CONTAINS.strip().lower()
    exact = set(ONLY_MODEL_NAMES or [])

    selected = []
    for item in jobs:
        if not item.get("run", True):
            continue
        if exact and item["model_name"] not in exact:
            continue
        if only_contains and only_contains not in item["model_name"].lower():
            continue
        selected.append(item)
    return selected

def existing_metadata_for_model(model_name: str) -> list[Path]:
    """Return only existing metadata matching this final test-set FLAN LoRA protocol."""
    safe_model = safe_filename(model_name)
    metadata_dir = LOCAL_PROJECT / "outputs" / "metadata"
    if not metadata_dir.exists():
        return []

    matches = []
    for path in sorted(metadata_dir.glob(f"run__finetuned__lora__{LABEL_MODE}__{safe_model}__*.json")):
        try:
            meta = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if (
            str(meta.get("eval_path", "")).replace("\\", "/").endswith(EVAL_PATH)
            and bool(meta.get("final_eval_only", False)) == bool(FINAL_EVAL_ONLY)
            and meta.get("model_type") == "flan_lora_finetuned_seq2seq_verbalizer"
        ):
            matches.append(path)
    return matches

def backup_outputs_to_drive() -> None:
    DRIVE_RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    src_outputs = LOCAL_PROJECT / "outputs"
    dst_outputs = DRIVE_RUN_OUTPUT_DIR / "outputs"

    if src_outputs.exists():
        if dst_outputs.exists():
            shutil.rmtree(dst_outputs)
        shutil.copytree(src_outputs, dst_outputs)
        print(f"Backed up outputs -> {dst_outputs}")
    else:
        print("No outputs directory yet; nothing to back up.")

def build_command(job: dict) -> list[str]:
    cmd = [
        sys.executable,
        FINETUNE_SCRIPT,
        "--model-name", job["model_name"],
        "--train-path", TRAIN_PATH,
        "--eval-path", EVAL_PATH,
        "--seed", str(SEED),
        "--max-length", str(MAX_LENGTH),
        "--epochs", str(EPOCHS),
        "--learning-rate", str(LEARNING_RATE),
        "--weight-decay", str(WEIGHT_DECAY),
        "--warmup-ratio", str(WARMUP_RATIO),
        "--train-batch-size", str(job["train_batch_size"]),
        "--eval-batch-size", str(job["eval_batch_size"]),
        "--grad-accum-steps", str(job["grad_accum_steps"]),
        "--label-mode", LABEL_MODE,
        "--best-metric", BEST_METRIC,
        "--preview-n", str(PREVIEW_N),
        "--warmup-path", WARMUP_PATH,
        "--warmup-n", str(WARMUP_N),
        "--run-tag", RUN_TAG,
        "--notes", f"Colab VS Code FLAN LoRA final test evaluation; run_tag={RUN_TAG}",
        "--lora-r", str(LORA_R),
        "--lora-alpha", str(LORA_ALPHA),
        "--lora-dropout", str(LORA_DROPOUT),
        "--lora-target-modules", LORA_TARGET_MODULES,
    ]

    if FINAL_EVAL_ONLY:
        cmd.append("--final-eval-only")
    if FP16:
        cmd.append("--fp16")
    if job.get("gradient_checkpointing", False):
        cmd.append("--gradient-checkpointing")
    if TRAIN_LIMIT is not None:
        cmd.extend(["--train-limit", str(TRAIN_LIMIT)])
    if EVAL_LIMIT is not None:
        cmd.extend(["--eval-limit", str(EVAL_LIMIT)])
    if WRITE_CURRENT:
        cmd.append("--write-current")
    if RUN_PARTICIPANT_SCORER:
        cmd.extend([
            "--run-participant-scorer",
            "--reference-dir", REFERENCE_DIR,
            "--score-split", SCORE_SPLIT,
        ])

    return cmd

def run_one_job(job: dict) -> dict:
    model_name = job["model_name"]
    safe_model = safe_filename(model_name)

    if SKIP_EXISTING:
        existing = existing_metadata_for_model(model_name)
        if existing:
            print(f"Skipping existing final run for {model_name}:")
            for path in existing:
                print("-", path)
            return {
                "model_name": model_name,
                "status": "skipped_existing",
                "metadata_files": [str(p) for p in existing],
            }

    cmd = build_command(job)
    print("\n" + "=" * 100)
    print(f"Running fine-tune final test eval: {model_name}")
    print("=" * 100)
    print("Command:", " ".join(cmd))

    if DRY_RUN:
        return {"model_name": model_name, "status": "dry_run", "command": cmd}

    DRIVE_RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    logs_dir = DRIVE_RUN_OUTPUT_DIR / "logs"
    logs_dir.mkdir(parents=True, exist_ok=True)
    log_path = logs_dir / f"{safe_model}.log"

    started_at = datetime.now().isoformat(timespec="seconds")
    with log_path.open("w", encoding="utf-8") as log_file:
        proc = subprocess.Popen(
            cmd,
            cwd=LOCAL_PROJECT,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            log_file.write(line)
        returncode = proc.wait()

    ended_at = datetime.now().isoformat(timespec="seconds")
    backup_outputs_to_drive()

    status = "success" if returncode == 0 else "failed"
    print(f"{status.upper()}: {model_name} with return code {returncode}")

    return {
        "model_name": model_name,
        "status": status,
        "returncode": returncode,
        "started_at": started_at,
        "ended_at": ended_at,
        "log_path": str(log_path),
    }

9) Optional smoke/dry run

In [9]:
print("DRY_RUN:", DRY_RUN)
print("ONLY_CONTAINS:", ONLY_CONTAINS)
print("ONLY_MODEL_NAMES:", ONLY_MODEL_NAMES)
print("FINETUNE_SCRIPT:", FINETUNE_SCRIPT)
print("TRAIN_PATH:", TRAIN_PATH)
print("EVAL_PATH:", EVAL_PATH)
print("FINAL_EVAL_ONLY:", FINAL_EVAL_ONLY)
print("LABEL_MODE:", LABEL_MODE)
print("EPOCHS:", EPOCHS)
print("LEARNING_RATE:", LEARNING_RATE)
print("LORA_R:", LORA_R)
print("LORA_TARGET_MODULES:", LORA_TARGET_MODULES)
print("RUN_PARTICIPANT_SCORER:", RUN_PARTICIPANT_SCORER)

selected = selected_jobs(FINETUNE_JOBS)
print(f"Selected {len(selected)} job(s):")
for job in selected:
    print(f"- {job['model_name']}")
    print("  ", " ".join(build_command(job)))

DRY_RUN: False
ONLY_CONTAINS: 
ONLY_MODEL_NAMES: []
FINETUNE_SCRIPT: finetune_flan_lora_final_eval_runtime_enhanced.py
TRAIN_PATH: data/SHROOM_dev-v2/val.model-agnostic.json
EVAL_PATH: data/SHROOM_test-labeled/test.model-agnostic.json
FINAL_EVAL_ONLY: True
LABEL_MODE: soft
EPOCHS: 4
LEARNING_RATE: 0.0002
LORA_R: 16
LORA_TARGET_MODULES: q,k,v,o
RUN_PARTICIPANT_SCORER: True
Selected 2 job(s):
- google/flan-t5-large
   /usr/bin/python3 finetune_flan_lora_final_eval_runtime_enhanced.py --model-name google/flan-t5-large --train-path data/SHROOM_dev-v2/val.model-agnostic.json --eval-path data/SHROOM_test-labeled/test.model-agnostic.json --seed 42 --max-length 512 --epochs 4 --learning-rate 0.0002 --weight-decay 0.0 --warmup-ratio 0.1 --train-batch-size 1 --eval-batch-size 1 --grad-accum-steps 4 --label-mode soft --best-metric rho --preview-n 2 --warmup-path data/SHROOM_trial-v1.1/trial-v1.json --warmup-n 10 --run-tag 20260514_164059 --notes Colab VS Code FLAN LoRA final test evaluation; run_

10) Run selected final fine-tuning/evaluation jobs and back up after each one

In [10]:
DRIVE_RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = DRIVE_RUN_OUTPUT_DIR / "run_manifest.json"
results = []

selected = selected_jobs(FINETUNE_JOBS)
if not selected:
    raise ValueError("No jobs selected. Check ONLY_CONTAINS / ONLY_MODEL_NAMES / run flags.")

for job in selected:
    result = run_one_job(job)
    results.append(result)

    with manifest_path.open("w", encoding="utf-8") as f:
        json.dump(
            {
                "run_tag": RUN_TAG,
                "drive_project": str(DRIVE_PROJECT),
                "local_project": str(LOCAL_PROJECT),
                "drive_run_output_dir": str(DRIVE_RUN_OUTPUT_DIR),
                "finetune_script": FINETUNE_SCRIPT,
                "train_path": TRAIN_PATH,
                "eval_path": EVAL_PATH,
                "final_eval_only": FINAL_EVAL_ONLY,
                "label_mode": LABEL_MODE,
                "epochs": EPOCHS,
                "learning_rate": LEARNING_RATE,
                "lora_r": LORA_R,
                "lora_alpha": LORA_ALPHA,
                "lora_dropout": LORA_DROPOUT,
                "lora_target_modules": LORA_TARGET_MODULES,
                "best_metric": BEST_METRIC,
                "warmup_path": WARMUP_PATH,
                "warmup_n": WARMUP_N,
                "seed": SEED,
                "write_current": WRITE_CURRENT,
                "run_participant_scorer": RUN_PARTICIPANT_SCORER,
                "results": results,
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

    if result["status"] == "failed" and not CONTINUE_ON_ERROR:
        raise RuntimeError(f"Stopping after failed run: {job['model_name']}")

print("\nBatch complete. Manifest:", manifest_path)
print(json.dumps(results, indent=2))


Running fine-tune final test eval: google/flan-t5-large
Command: /usr/bin/python3 finetune_flan_lora_final_eval_runtime_enhanced.py --model-name google/flan-t5-large --train-path data/SHROOM_dev-v2/val.model-agnostic.json --eval-path data/SHROOM_test-labeled/test.model-agnostic.json --seed 42 --max-length 512 --epochs 4 --learning-rate 0.0002 --weight-decay 0.0 --warmup-ratio 0.1 --train-batch-size 1 --eval-batch-size 1 --grad-accum-steps 4 --label-mode soft --best-metric rho --preview-n 2 --warmup-path data/SHROOM_trial-v1.1/trial-v1.json --warmup-n 10 --run-tag 20260514_164059 --notes Colab VS Code FLAN LoRA final test evaluation; run_tag=20260514_164059 --lora-r 16 --lora-alpha 32 --lora-dropout 0.05 --lora-target-modules q,k,v,o --final-eval-only --write-current --run-participant-scorer --reference-dir data/SHROOM_test-labeled --score-split test
2026-05-14 16:43:36.907253: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical

11) Summarize final-test metadata files found in outputs/metadata

In [11]:
import json
from pathlib import Path

metadata_dir = LOCAL_PROJECT / "outputs" / "metadata"
rows = []

if metadata_dir.exists():
    for path in sorted(metadata_dir.glob("run__finetuned__lora__*.json")):
        try:
            meta = json.loads(path.read_text(encoding="utf-8"))
        except Exception as exc:
            print("Could not read", path, exc)
            continue

        if meta.get("model_type") != "flan_lora_finetuned_seq2seq_verbalizer":
            continue
        if not bool(meta.get("final_eval_only", False)):
            continue
        if not str(meta.get("eval_path", "")).replace("\\", "/").endswith(EVAL_PATH):
            continue

        comp = meta.get("computational_cost", {})
        hp = meta.get("hyperparameters", {})
        final_metrics = meta.get("final_prediction_metrics", {})
        participant_scores = meta.get("participant_scores", {}) or {}
        lora = meta.get("lora", {}) or {}

        rows.append({
            "base_model_name": meta.get("base_model_name"),
            "parameter_count": comp.get("parameter_count"),
            "trainable_parameter_count": comp.get("trainable_parameter_count"),
            "trainable_fraction": comp.get("trainable_parameter_fraction"),
            "label_mode": meta.get("label_mode"),
            "split_mode": meta.get("split_mode"),
            "final_eval_only": meta.get("final_eval_only"),
            "checkpoint_selection": meta.get("checkpoint_selection"),
            "num_train": meta.get("num_train_examples"),
            "num_eval": meta.get("num_eval_examples"),
            "epochs": hp.get("epochs"),
            "lr": hp.get("learning_rate"),
            "effective_bs": hp.get("effective_train_batch_size"),
            "grad_ckpt": hp.get("gradient_checkpointing"),
            "lora_r": lora.get("r"),
            "lora_targets": ",".join(lora.get("target_modules", [])) if isinstance(lora.get("target_modules"), list) else lora.get("target_modules"),
            "test_acc": final_metrics.get("accuracy") or participant_scores.get("acc_agnostic"),
            "test_rho": final_metrics.get("rho") or participant_scores.get("rho_agnostic"),
            "train_runtime_s": comp.get("total_training_runtime_seconds"),
            "mean_latency_s": comp.get("mean_inference_latency_seconds_per_example"),
            "score_path": meta.get("score_path"),
            "metadata_file": str(path),
        })

if not rows:
    print("No matching FLAN LoRA final-test metadata rows found yet.")
else:
    try:
        import pandas as pd
        df = pd.DataFrame(rows)
        display(df.sort_values(["parameter_count", "base_model_name"], na_position="last"))
    except Exception:
        for row in rows:
            print(row)

backup_outputs_to_drive()

No matching FLAN LoRA final-test metadata rows found yet.
Backed up outputs -> /content/drive/MyDrive/thesis_colab/outputs_flan_lora_finetune_test_colab_vscode - A100, rerun/20260514_164059/outputs


12) Optional: show score.py output files when participant scorer is enabled

In [ ]:
from pathlib import Path

score_dir = LOCAL_PROJECT / "outputs" / "scores"
if not score_dir.exists():
    print("No outputs/scores directory found.")
else:
    score_files = sorted(score_dir.glob("finetuned_scores__lora__*.txt")) + sorted(score_dir.glob("*flan*.txt"))
    if not score_files:
        print("No FLAN LoRA score text files found. This is expected if RUN_PARTICIPANT_SCORER = False.")
    else:
        for path in score_files[-10:]:
            print("\n" + "=" * 80)
            print(path)
            print("=" * 80)
            print(path.read_text(encoding="utf-8"))

13) Notes

In [ ]:
# This notebook is for the final labeled-test evaluation of FLAN LoRA rungs.
#
# The key difference from the internal-split runner is --final-eval-only: the script trains for
# the fixed number of epochs and only evaluates on the test file after training. This avoids
# selecting the best epoch/checkpoint using test-set performance.
#
# Recommended usage:
# 1. Keep DRY_RUN = True and run cells top-to-bottom to inspect commands.
# 2. Set ONLY_MODEL_NAMES = ["google/flan-t5-small"] and DRY_RUN = False for a one-model smoke test.
# 3. Then set ONLY_MODEL_NAMES = [] and keep XL run=False for small/base/large final runs.
# 4. To include XL later, change the XL job's run flag to True and use a high-memory GPU.